# Create GeneralTriplet hard negatives for Kaggle

Add the SQLite database and the fine-tuned model as Kaggle inputs, update the two input paths below, then run both code cells. The output is `/kaggle/working/general_triplet.db`.

In [ ]:
import json
import os
import sqlite3
from pathlib import Path

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# Update these two paths to match the Kaggle input dataset slugs.
INPUT_DB_PATH = Path("/kaggle/input/YOUR-GENERAL-DATABASE/general_export.db")
MODEL_PATH = Path("/kaggle/input/YOUR-FINETUNED-MODEL/checkpoint-40236")

OUTPUT_DB_PATH = Path("/kaggle/working/general_triplet.db")
ARTIFACT_DIR = Path("/kaggle/working/general_hard_negative_index")
INDEX_PATH = ARTIFACT_DIR / "general.faiss"
DATA_IDS_PATH = ARTIFACT_DIR / "general_data_ids.npy"
METADATA_PATH = ARTIFACT_DIR / "metadata.json"

EMBEDDING_BATCH_SIZE = 128
DATABASE_BATCH_SIZE = 256
INDEX_TRAINING_SAMPLE_SIZE = 200_000
MIN_TRAINING_SAMPLE_SIZE = 50_000
INDEX_NLIST = 4_096
INDEX_PQ_M = 64
INDEX_NPROBE = 32
SEARCH_TOP_K = 16
PROGRESS_EVERY = 10_000
MAX_ROWS = None  # Set a small value, for example 10_000, to debug first.
REBUILD_INDEX = False  # Set True only after changing the input DB or model.
REQUIRE_GPU = True  # Full-scale mining should not run on CPU.

if not INPUT_DB_PATH.is_file():
    raise FileNotFoundError(f"Input database does not exist: {INPUT_DB_PATH}")
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f"Model directory does not exist: {MODEL_PATH}")

print(f"Input DB: {INPUT_DB_PATH}")
print(f"Model: {MODEL_PATH}")
print(f"Output DB: {OUTPUT_DB_PATH}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. In Kaggle, enable Settings > Accelerator > GPU, "
        "then restart the kernel. Installing faiss-gpu alone does not enable PyTorch CUDA."
    )

In [ ]:
def valid_where() -> str:
    return "anchor IS NOT NULL AND positive IS NOT NULL AND length(trim(anchor)) > 0 AND length(trim(positive)) > 0"


def encode_texts(
    model: SentenceTransformer,
    texts: list[str],
    prefix: str,
    description: str | None = None,
) -> np.ndarray:
    batches = range(0, len(texts), EMBEDDING_BATCH_SIZE)
    if description is not None:
        batches = tqdm(batches, desc=description, unit="batch")

    embeddings = []
    for start in batches:
        batch = texts[start : start + EMBEDDING_BATCH_SIZE]
        batch_embeddings = model.encode(
            [f"{prefix}: {text}" for text in batch],
            batch_size=EMBEDDING_BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        embeddings.append(np.asarray(batch_embeddings, dtype=np.float32))

    return np.vstack(embeddings)


def encode_passages(
    model: SentenceTransformer,
    texts: list[str],
    description: str | None = None,
) -> np.ndarray:
    return encode_texts(model, texts, prefix="passage", description=description)


def encode_queries(model: SentenceTransformer, texts: list[str]) -> np.ndarray:
    return encode_texts(model, texts, prefix="query")


def get_source_stats(connection: sqlite3.Connection) -> tuple[int, int]:
    row_count = connection.execute(
        f"SELECT COUNT(*), MAX(length(data_id)) FROM general WHERE {valid_where()}"
    ).fetchone()
    if row_count[0] == 0 or row_count[1] is None:
        raise ValueError("No valid rows were found in the general table.")
    return int(row_count[0]), int(row_count[1])


def fetch_source_batch(
    connection: sqlite3.Connection,
    last_data_id: str | None,
) -> list[dict]:
    query = f"""
        SELECT data_id, source, title, topic, anchor, positive
        FROM general
        WHERE {valid_where()}
    """
    parameters: list[object] = []
    if last_data_id is not None:
        query += " AND data_id > ?"
        parameters.append(last_data_id)
    query += " ORDER BY data_id LIMIT ?"
    parameters.append(DATABASE_BATCH_SIZE)

    cursor = connection.execute(query, parameters)
    return [dict(row) for row in cursor.fetchall()]


def artifact_paths() -> tuple[Path, ...]:
    return INDEX_PATH, DATA_IDS_PATH, METADATA_PATH


def building_path(path: Path) -> Path:
    return path.with_name(f"{path.name}.building")


def clear_index_artifacts() -> None:
    paths = (*artifact_paths(), *(building_path(path) for path in artifact_paths()))
    for path in paths:
        if path.exists():
            path.unlink()


def expected_metadata(
    source_count: int,
    max_data_id_length: int,
    embedding_dimension: int,
) -> dict:
    return {
        "source_count": source_count,
        "max_data_id_length": max_data_id_length,
        "embedding_dimension": embedding_dimension,
        "index_nlist": INDEX_NLIST,
        "index_pq_m": INDEX_PQ_M,
        "index_nprobe": INDEX_NPROBE,
        "model_path": str(MODEL_PATH),
    }


def load_saved_index(metadata: dict):
    files_exist = [path.exists() for path in artifact_paths()]
    building_files = [path for path in artifact_paths() if building_path(path).exists()]
    if building_files:
        raise RuntimeError("Interrupted index artifacts found. Set REBUILD_INDEX = True to replace them.")
    if not any(files_exist):
        return None
    if not all(files_exist):
        raise RuntimeError("Index artifacts are incomplete. Set REBUILD_INDEX = True to recreate them.")
    if json.loads(METADATA_PATH.read_text()) != metadata:
        raise RuntimeError("Index metadata changed. Set REBUILD_INDEX = True to recreate it.")

    index = faiss.read_index(str(INDEX_PATH))
    index.nprobe = INDEX_NPROBE
    data_ids = np.load(DATA_IDS_PATH, mmap_mode="r")
    if index.ntotal != metadata["source_count"] or len(data_ids) != index.ntotal:
        raise RuntimeError("FAISS index and ID map sizes differ. Set REBUILD_INDEX = True.")
    print(f"Loaded FAISS index: {index.ntotal:,} vectors")
    return index, data_ids


def sample_training_texts(connection: sqlite3.Connection, source_count: int) -> list[str]:
    sample_divisor = max(1, source_count // (INDEX_TRAINING_SAMPLE_SIZE * 2))
    print("Reading a random FAISS training sample from SQLite...")
    rows = connection.execute(
        f"""
        SELECT positive
        FROM general
        WHERE {valid_where()}
          AND ((random() & 9223372036854775807) % ?) = 0
        LIMIT ?
        """,
        (sample_divisor, INDEX_TRAINING_SAMPLE_SIZE),
    ).fetchall()
    texts = [row[0] for row in rows]
    if len(texts) < MIN_TRAINING_SAMPLE_SIZE:
        raise RuntimeError(
            f"FAISS training sample is too small: {len(texts):,} rows. Run the cell again."
        )
    print(f"FAISS training sample: {len(texts):,} passages")
    return texts


def build_index(
    connection: sqlite3.Connection,
    model: SentenceTransformer,
    metadata: dict,
):
    source_count = metadata["source_count"]
    max_data_id_length = metadata["max_data_id_length"]
    dimension = metadata["embedding_dimension"]
    if dimension % INDEX_PQ_M != 0:
        raise ValueError(f"Embedding dimension {dimension} is not divisible by {INDEX_PQ_M}.")

    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    index_building = building_path(INDEX_PATH)
    data_ids_building = building_path(DATA_IDS_PATH)
    metadata_building = building_path(METADATA_PATH)

    training_texts = sample_training_texts(connection, source_count)
    training_embeddings = encode_passages(
        model,
        training_texts,
        description="Encoding FAISS training sample",
    )
    del training_texts
    print(f"Training FAISS with {len(training_embeddings):,} embeddings...")

    index = faiss.IndexIVFPQ(
        faiss.IndexFlatIP(dimension),
        dimension,
        INDEX_NLIST,
        INDEX_PQ_M,
        8,
        faiss.METRIC_INNER_PRODUCT,
    )
    index.train(training_embeddings)
    index.nprobe = INDEX_NPROBE

    data_ids = np.lib.format.open_memmap(
        data_ids_building,
        mode="w+",
        dtype=f"S{max_data_id_length}",
        shape=(source_count,),
    )

    last_data_id = None
    offset = 0
    while True:
        rows = fetch_source_batch(connection, last_data_id)
        if not rows:
            break

        index.add(encode_passages(model, [row["positive"] for row in rows]))
        data_ids[offset : offset + len(rows)] = [row["data_id"].encode("utf-8") for row in rows]
        offset += len(rows)
        last_data_id = rows[-1]["data_id"]

        if offset % PROGRESS_EVERY < len(rows):
            print(f"Indexed: {offset:,}/{source_count:,}")

    if offset != source_count:
        raise RuntimeError(f"Indexed {offset:,} rows, expected {source_count:,}.")

    data_ids.flush()
    faiss.write_index(index, str(index_building))
    metadata_building.write_text(json.dumps(metadata, indent=2))
    os.replace(data_ids_building, DATA_IDS_PATH)
    os.replace(index_building, INDEX_PATH)
    os.replace(metadata_building, METADATA_PATH)

    print(f"Saved FAISS index: {index.ntotal:,} vectors")
    return index, np.load(DATA_IDS_PATH, mmap_mode="r")


def get_data_id(data_ids: np.ndarray, position: int) -> str:
    return data_ids[position].tobytes().decode("utf-8").rstrip("\x00")


def get_existing_ids(connection: sqlite3.Connection, data_ids: list[str]) -> set[str]:
    if not data_ids:
        return set()
    placeholders = ",".join("?" for _ in data_ids)
    rows = connection.execute(
        f"SELECT data_id FROM general_triplet WHERE data_id IN ({placeholders})",
        data_ids,
    )
    return {row[0] for row in rows}


def fetch_candidate_positives(
    connection: sqlite3.Connection,
    data_ids: set[str],
) -> dict[str, str]:
    values = list(data_ids)
    positives = {}
    for start in range(0, len(values), 900):
        batch = values[start : start + 900]
        placeholders = ",".join("?" for _ in batch)
        rows = connection.execute(
            f"SELECT data_id, positive FROM general WHERE data_id IN ({placeholders})",
            batch,
        )
        positives.update({data_id: positive for data_id, positive in rows if positive and positive.strip()})
    return positives


def create_output_table(connection: sqlite3.Connection) -> None:
    connection.execute("""
        CREATE TABLE IF NOT EXISTS general_triplet (
            data_id TEXT PRIMARY KEY,
            source TEXT,
            title TEXT,
            topic TEXT,
            anchor TEXT NOT NULL,
            positive TEXT NOT NULL,
            hard_negative TEXT NOT NULL
        )
    """)
    connection.commit()


def build_triplets(
    rows: list[dict],
    positions: np.ndarray,
    data_ids: np.ndarray,
    candidate_positives: dict[str, str],
) -> tuple[list[tuple], int]:
    triplets = []
    skipped = 0
    for row, neighbors in zip(rows, positions, strict=True):
        hard_negative = None
        for position in neighbors:
            if position < 0:
                continue
            candidate_id = get_data_id(data_ids, int(position))
            candidate_positive = candidate_positives.get(candidate_id)
            if candidate_id == row["data_id"]:
                continue
            if candidate_positive is None or candidate_positive == row["positive"]:
                continue
            hard_negative = candidate_positive
            break

        if hard_negative is None:
            skipped += 1
            continue

        triplets.append((
            row["data_id"],
            row["source"],
            row["title"],
            row["topic"],
            row["anchor"],
            row["positive"],
            hard_negative,
        ))
    return triplets, skipped


source_connection = sqlite3.connect(f"file:{INPUT_DB_PATH}?mode=ro", uri=True)
source_connection.row_factory = sqlite3.Row
output_connection = sqlite3.connect(OUTPUT_DB_PATH)
output_connection.execute("PRAGMA journal_mode = WAL")
output_connection.execute("PRAGMA synchronous = NORMAL")
output_connection.execute("PRAGMA temp_store = MEMORY")

try:
    columns = {row[1] for row in source_connection.execute("PRAGMA table_info(general)")}
    required_columns = {"data_id", "source", "title", "topic", "anchor", "positive"}
    missing_columns = required_columns.difference(columns)
    if missing_columns:
        raise ValueError(f"Input general table is missing columns: {sorted(missing_columns)}")

    create_output_table(output_connection)
    source_count, max_data_id_length = get_source_stats(source_connection)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading model on: {device}")
    model = SentenceTransformer(str(MODEL_PATH), device=device, local_files_only=True)
    metadata = expected_metadata(
        source_count,
        max_data_id_length,
        model.get_embedding_dimension(),
    )

    if REBUILD_INDEX:
        clear_index_artifacts()
    saved_index = load_saved_index(metadata)
    index, data_ids = saved_index or build_index(source_connection, model, metadata)

    existing_count = output_connection.execute("SELECT COUNT(*) FROM general_triplet").fetchone()[0]
    print(f"Source records: {source_count:,}")
    print(f"Existing output records: {existing_count:,}")

    insert_query = """
        INSERT OR IGNORE INTO general_triplet (
            data_id, source, title, topic, anchor, positive, hard_negative
        ) VALUES (?, ?, ?, ?, ?, ?, ?)
    """

    last_data_id = None
    processed = 0
    inserted = 0
    skipped_without_negative = 0

    while True:
        rows = fetch_source_batch(source_connection, last_data_id)
        if not rows:
            break
        last_data_id = rows[-1]["data_id"]

        completed_ids = get_existing_ids(output_connection, [row["data_id"] for row in rows])
        rows = [row for row in rows if row["data_id"] not in completed_ids]
        if not rows:
            continue

        query_embeddings = encode_queries(model, [row["anchor"] for row in rows])
        _, positions = index.search(query_embeddings, SEARCH_TOP_K)
        candidate_ids = {
            get_data_id(data_ids, int(position))
            for neighbors in positions
            for position in neighbors
            if position >= 0
        }
        candidate_positives = fetch_candidate_positives(source_connection, candidate_ids)
        triplets, skipped = build_triplets(rows, positions, data_ids, candidate_positives)

        output_connection.executemany(insert_query, triplets)
        output_connection.commit()

        processed += len(rows)
        inserted += len(triplets)
        skipped_without_negative += skipped

        if MAX_ROWS is not None and processed >= MAX_ROWS:
            break
        if processed % PROGRESS_EVERY < len(rows):
            print(
                f"Processed: {processed:,}; inserted this run: {inserted:,}; "
                f"without hard negative: {skipped_without_negative:,}"
            )

    row_count = output_connection.execute("SELECT COUNT(*) FROM general_triplet").fetchone()[0]
    integrity = output_connection.execute("PRAGMA integrity_check").fetchone()[0]
    print(f"Finished. Output rows: {row_count:,}; integrity: {integrity}")
finally:
    output_connection.commit()
    output_connection.execute("PRAGMA wal_checkpoint(TRUNCATE)")
    output_connection.execute("PRAGMA journal_mode = DELETE")
    output_connection.close()
    source_connection.close()

print(f"Kaggle output file: {OUTPUT_DB_PATH}")
print(f"Output size: {OUTPUT_DB_PATH.stat().st_size / 1024**3:.2f} GB")